In [1]:
# =========================
# 1. 기본 설정 및 라이브러리 임포트
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# 재현성을 위한 시드 설정
np.random.seed(42)
tf.random.set_seed(42)

print("라이브러리 로딩 완료")


라이브러리 로딩 완료


## ✅ 데이터 로딩은 기존 코드(사용자 파일) 그대로 사용하세요

이 노트북은 **`train_df`, `test_df`, `rul_truth`가 이미 로딩되어 있다**는 가정 하에,
그 이후(전처리/특성/모델)만 **리더보드 점수 개선용으로 업그레이드**한 버전입니다.

핵심 개선 포인트:
- **운영 조건(Setting) 기반 클러스터링 후, 클러스터별 스케일링** (FD002 필수급)
- **엔진 단위(Group)로 Train/Val 분리** → 누수 방지 + 일반화 개선
- **차분/EMA/롤링 통계** 등 시계열 특성 강화
- **NASA Score와 유사한 형태의 손실 함수**로 직접 최적화
- Conv1D + BiLSTM + MultiHeadAttention 구조로 패턴 포착 강화


In [2]:
# =========================
# 2. 데이터 로딩
# =========================

# Kaggle 환경에 맞는 데이터 경로 설정
DATA_DIR = r"C:\\Users\\yuzhd\\github\\DataScience\\scikit-learn\\scikit-learn\\data\\CMaps"

# 컬럼 이름 정의
index_names = ['unit_number', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names = [f's_{i}' for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

# 데이터 로딩
train_df = pd.read_csv(f'{DATA_DIR}/train_FD002.txt', sep=r'\s+', header=None, names=col_names)
test_df = pd.read_csv(f'{DATA_DIR}/test_FD002.txt', sep=r'\s+', header=None, names=col_names)
rul_truth = pd.read_csv(f'{DATA_DIR}/RUL_FD002.txt', sep=r'\s+', header=None, names=['RUL'])

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"RUL truth shape: {rul_truth.shape}")
print(f"Number of engines in train: {train_df['unit_number'].nunique()}")
print(f"Number of engines in test: {test_df['unit_number'].nunique()}")

# 기본 데이터 확인
display(train_df.head())


Train shape: (53759, 26)
Test shape: (33991, 26)
RUL truth shape: (259, 1)
Number of engines in train: 260
Number of engines in test: 259


,unit_number,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,...,183.06,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,...,130.42,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,...,164.22,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,...,130.72,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,...,164.31,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286


In [3]:
# =========================
# 3. 데이터 전처리 - RUL 계산 (개선 버전)
# =========================

def add_rul_to_train(df, max_rul=200):
    """훈련 데이터에 RUL 추가 + 초기 정상 구간 cap"""
    max_cycles = df.groupby('unit_number')['time_cycles'].max().rename('max_cycle').reset_index()
    out = df.merge(max_cycles, on='unit_number', how='left')
    out['RUL'] = (out['max_cycle'] - out['time_cycles']).clip(lower=0, upper=max_rul)
    out.drop(columns=['max_cycle'], inplace=True)
    return out

# ✅ 이미 RUL이 있으면 그대로 사용, 없으면 생성
if 'RUL' not in train_df.columns:
    train_df = add_rul_to_train(train_df, max_rul=200)

print('Train RUL stats:')
display(train_df['RUL'].describe())


Train RUL stats:


count    53759.000000
mean       104.200320
std         61.417972
min          0.000000
25%         51.000000
50%        103.000000
75%        156.000000
max        200.000000
Name: RUL, dtype: float64

In [4]:
# =========================
# 4. FD002 핵심: 운영 조건(Setting) 클러스터링 + 클러스터별 정규화
# =========================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

setting_cols = [c for c in train_df.columns if c.startswith('setting_')]
sensor_cols  = [c for c in train_df.columns if c.startswith('s_')]
base_feature_cols = setting_cols + sensor_cols

# FD002는 운영 조건이 여러 개(통상 6개 근처)로 알려져 있어 K=6부터 시도하는 경우가 많음
N_OP_COND = 6

kmeans = KMeans(n_clusters=N_OP_COND, random_state=42, n_init='auto')
train_df['op_cond'] = kmeans.fit_predict(train_df[setting_cols])
test_df['op_cond']  = kmeans.predict(test_df[setting_cols])
# op_cluster 별칭(뒤 단계 호환)
train_df['op_cluster'] = train_df['op_cond']
test_df['op_cluster']  = test_df['op_cond']


# ✅ 클러스터별로 센서/세팅을 표준화 (train로 fit, train/test에 동일 적용)
scalers = {}
for k in range(N_OP_COND):
    idx_tr = train_df['op_cond'] == k
    scaler = StandardScaler()
    train_df.loc[idx_tr, base_feature_cols] = scaler.fit_transform(train_df.loc[idx_tr, base_feature_cols])

    idx_te = test_df['op_cond'] == k
    if idx_te.any():
        test_df.loc[idx_te, base_feature_cols] = scaler.transform(test_df.loc[idx_te, base_feature_cols])

    scalers[k] = scaler

print('op_cond counts (train):')
display(train_df['op_cond'].value_counts().sort_index())


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\yuzhd\miniconda3\envs\DS\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\yuzhd\miniconda3\envs\DS\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\yuzhd\miniconda3\envs\DS\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 4: invalid start byte


op_cond counts (train):


op_cond
0    13458
1     8122
2     8002
3     8044
4     8096
5     8037
Name: count, dtype: int64

In [5]:
# =========================
# 5. 특성 엔지니어링 (강화): diff + EMA + rolling stats
# =========================
import numpy as np

def add_time_series_features(df, sensor_cols, windows=(5, 10, 20), ema_spans=(10, 20)):
    out = df.copy()

    # 1) 1-step difference (변화량)
    for c in sensor_cols:
        out[f'{c}_diff1'] = out.groupby('unit_number')[c].diff().fillna(0)

    # 2) EMA (추세)
    for c in sensor_cols:
        for span in ema_spans:
            out[f'{c}_ema_{span}'] = (
                out.groupby('unit_number')[c]
                .transform(lambda x: x.ewm(span=span, adjust=False).mean())
            )

    # 3) rolling mean / std (변동성)
    for c in sensor_cols:
        for w in windows:
            out[f'{c}_rm_{w}'] = (
                out.groupby('unit_number')[c]
                .transform(lambda x: x.rolling(window=w, min_periods=1).mean())
            )
            out[f'{c}_rs_{w}'] = (
                out.groupby('unit_number')[c]
                .transform(lambda x: x.rolling(window=w, min_periods=1).std())
                .fillna(0)
            )
    return out

print('Adding time-series features...')
train_df = add_time_series_features(train_df, sensor_cols)
test_df  = add_time_series_features(test_df, sensor_cols)

# 최종 특성 컬럼
feature_cols = [c for c in train_df.columns if (c.startswith('setting_') or c.startswith('s_')) or ('_diff1' in c) or ('_ema_' in c) or ('_rm_' in c) or ('_rs_' in c)]
# op_cond도 입력으로 넣고 싶으면 원-핫을 추가할 수 있지만, 여기서는 숫자 그대로 사용
feature_cols += ['op_cond']

print('n_features:', len(feature_cols))


Adding time-series features...
n_features: 214


In [6]:
# =========================
# 6. 시퀀스 생성 (개선 v3): 메타(unit, cycle) 포함 + Test는 last-k 윈도우 앙상블
# =========================
import numpy as np

SEQUENCE_LENGTH = 50
TEST_LAST_K = 7   # 마지막 K개 사이클 윈도우 예측을 모아 median/mean 앙상블

def _pad_left(arr, target_len):
    # arr: (T, F)
    if len(arr) >= target_len:
        return arr[-target_len:]
    pad = np.repeat(arr[:1], repeats=(target_len - len(arr)), axis=0)
    return np.concatenate([pad, arr], axis=0)

def make_train_sequences(df, feature_cols, seq_len=50, target_col='RUL'):
    """모든 타임스텝에 대해 (seq, y, unit, cycle) 생성"""
    X, y, units, cycles = [], [], [], []
    for unit, g in df.groupby('unit_number'):
        g = g.sort_values('time_cycles')
        feats = g[feature_cols].to_numpy(dtype=np.float32)
        t = g[target_col].to_numpy(dtype=np.float32)
        c = g['time_cycles'].to_numpy(dtype=np.int32)

        if len(g) < seq_len:
            X.append(_pad_left(feats, seq_len))
            y.append(t[-1])
            units.append(unit)
            cycles.append(int(c[-1]))
        else:
            for i in range(seq_len - 1, len(g)):
                X.append(feats[i-seq_len+1:i+1])
                y.append(t[i])
                units.append(unit)
                cycles.append(int(c[i]))

    return np.stack(X), np.array(y), np.array(units), np.array(cycles)

def make_test_sequences_lastk(df, feature_cols, seq_len=50, last_k=7):
    """각 unit의 마지막 last_k 사이클을 끝점으로 하는 윈도우들을 생성.
    반환:
      X: (sum_k, seq_len, F)
      units: (sum_k,)
      cycles: (sum_k,)  # 윈도우 끝 cycle
    """
    X, units, cycles = [], [], []
    for unit, g in df.groupby('unit_number'):
        g = g.sort_values('time_cycles')
        feats = g[feature_cols].to_numpy(dtype=np.float32)
        c = g['time_cycles'].to_numpy(dtype=np.int32)

        T = len(g)
        # 끝점 index들: last_k개 (T-1, T-2, ...)
        end_idxs = list(range(T-1, max(-1, T-1-last_k), -1))
        for ei in end_idxs:
            window = feats[:ei+1]  # 0..ei
            X.append(_pad_left(window, seq_len))
            units.append(unit)
            cycles.append(int(c[ei]))
    return np.stack(X), np.array(units), np.array(cycles)

X_all, y_all, groups, cycles_all = make_train_sequences(train_df, feature_cols, SEQUENCE_LENGTH, 'RUL')

# Test: last-k 윈도우로 예측을 더 안정화
X_test_k, test_units_k, test_cycles_k = make_test_sequences_lastk(test_df, feature_cols, SEQUENCE_LENGTH, TEST_LAST_K)

print('X_all:', X_all.shape, 'y_all:', y_all.shape)
print('X_test_k:', X_test_k.shape, 'test_units_k:', test_units_k.shape, 'test_cycles_k:', test_cycles_k.shape)


X_all: (41019, 50, 214) y_all: (41019,)
X_test_k: (1813, 50, 214) test_units_k: (1813,) test_cycles_k: (1813,)


In [7]:
# =========================
# 7. 엔진 단위(Group) Train/Val 분리 (누수 방지) + 메타 유지
# =========================
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, val_idx = next(gss.split(X_all, y_all, groups=groups))

X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]

groups_tr, groups_val = groups[tr_idx], groups[val_idx]
cycles_tr, cycles_val = cycles_all[tr_idx], cycles_all[val_idx]

print('Train:', X_tr.shape, y_tr.shape, 'units:', len(np.unique(groups_tr)))
print('Val  :', X_val.shape, y_val.shape, 'units:', len(np.unique(groups_val)))


Train: (33272, 50, 214) (33272,) units: 208
Val  : (7747, 50, 214) (7747,) units: 52


In [8]:
# =========================
# 8. 모델 (v3): Conv1D + BiLSTM + MultiHeadAttention (함수로만 정의)
# =========================
import tensorflow as tf
from tensorflow.keras import layers, Model

def make_model(input_shape):
    seq_len, n_feat = input_shape[0], input_shape[1]
    inp = layers.Input(shape=(seq_len, n_feat))

    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)

    # Self-attention
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='linear')(x)

    return Model(inp, out)

print('Model fn ready. input shape example:', X_tr.shape[1:], '(after cell 7)')


Model fn ready. input shape example: (50, 214) (after cell 7)


In [9]:
# =========================
# 8.5. (필수) NASA Score / MAX_RUL / Loss 정의 (학습 전에 필요)
# =========================
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def nasa_score(y_true, y_pred):
    """NASA scoring function (C-MAPSS style).
    d = y_pred - y_true
    d < 0 (under-pred): exp(-d/13)-1
    d > 0 (over-pred) : exp(d/10)-1  (과대예측이 훨씬 더 아픔)
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    d = y_pred - y_true
    s = np.where(d < 0, np.exp(-d/13.0) - 1.0, np.exp(d/10.0) - 1.0)
    return float(np.sum(s))

# FD002 test RUL 정답이 194까지 존재 → cap을 200으로 확장
MAX_RUL = 200

def asym_weighted_huber(y_true, y_pred, delta=10.0, over_penalty=3.0, low_rul_boost=3.0):
    """비대칭 + 낮은 RUL 가중 Huber loss.
    - 과대예측(over): penalty를 크게
    - 낮은 RUL(고장 임박) 구간: 가중을 더 줘서 NASA Score 폭발 방지
    """
    y_true = tf.cast(tf.reshape(y_true, (-1,)), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, (-1,)), tf.float32)

    err = y_pred - y_true
    abs_err = tf.abs(err)

    huber = tf.where(
        abs_err <= delta,
        0.5 * tf.square(err),
        delta * (abs_err - 0.5 * delta)
    )

    # over-pred (err>0) penalty
    over_mask = tf.cast(err > 0.0, tf.float32)
    asym_w = 1.0 + over_mask * (over_penalty - 1.0)

    # low RUL boost: y_true 낮을수록 weight↑
    low_w = 1.0 + (tf.maximum(0.0, float(MAX_RUL) - y_true) / float(MAX_RUL)) * low_rul_boost

    return tf.reduce_mean(huber * asym_w * low_w)

print('✅ prep ready: MAX_RUL=', MAX_RUL)


✅ prep ready: MAX_RUL= 200


In [10]:
# =========================
# 9. 학습 (v3): seed 앙상블 + val_nasa 기준으로 best seed 선택/저장 (FIXED)
# =========================
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEEDS = [11, 22, 33]

def set_seed(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)

# (중요) val NASA Score를 계산하는 콜백 (전역변수 의존 제거)
class NasaScoreCallback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, max_rul=130):
        super().__init__()
        self.Xv = X_val
        self.yv = y_val
        self.max_rul = max_rul
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.Xv, verbose=0).reshape(-1)
        y_pred = np.clip(y_pred, 0, self.max_rul)
        score = nasa_score(self.yv, y_pred)
        self.history.append(score)
        if logs is not None:
            logs['val_nasa'] = score
        print(f' — val_nasa: {score:,.2f}')

def build_model():
    if 'make_model' not in globals():
        raise NameError("make_model이 정의되어 있지 않습니다. (cell 8을 먼저 실행하세요)")
    return make_model(input_shape=X_tr.shape[1:])

# 필수 객체 존재 체크 (셀 순서 꼬임 방지용)
missing = [name for name in ['MAX_RUL','asym_weighted_huber','nasa_score'] if name not in globals()]
if missing:
    raise NameError(f"학습 전에 필요한 항목이 없습니다: {missing}. (cell 8.5를 먼저 실행하세요)")

best = {'seed': None, 'val_nasa': float('inf'), 'model': None, 'history': None}

for sd in SEEDS:
    print('\n' + '='*70)
    print('Training seed =', sd)
    print('='*70)
    set_seed(sd)

    model = build_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=asym_weighted_huber
    )

    nasa_cb = NasaScoreCallback(X_val, y_val, max_rul=MAX_RUL)

    callbacks = [
        nasa_cb,
        EarlyStopping(monitor='val_nasa', mode='min', patience=10, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_nasa', mode='min', factor=0.5, patience=4, min_lr=1e-5, verbose=1)
    ]

    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )

    val_nasa_best = float(np.min(nasa_cb.history))
    print(f'[seed {sd}] best val_nasa = {val_nasa_best:,.2f}')

    if val_nasa_best < best['val_nasa']:
        best.update({'seed': sd, 'val_nasa': val_nasa_best, 'model': model, 'history': hist})

print('\n✅ BEST seed:', best['seed'], 'best val_nasa:', f"{best['val_nasa']:,.2f}")
model = best['model']



Training seed = 11
Epoch 1/60
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 1092.2032 — val_nasa: 588,738.00
130/130 ━━━━━━━━━━━━━━━━━━━━ 15s 92ms/step - loss: 739.1450 - val_loss: 410.0924 - val_nasa: 588738.0000 - learning_rate: 0.0010
Epoch 2/60
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 406.8620 — val_nasa: 438,361.50
130/130 ━━━━━━━━━━━━━━━━━━━━ 12s 89ms/step - loss: 380.6060 - val_loss: 360.8792 - val_nasa: 438361.5000 - learning_rate: 0.0010
Epoch 3/60
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 335.8479 — val_nasa: 567,859.25
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - loss: 309.1924 - val_loss: 330.5111 - val_nasa: 567859.2500 - learning_rate: 0.0010
Epoch 4/60
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 277.5674 — val_nasa: 311,440.25
130/130 ━━━━━━━━━━━━━━━━━━━━ 12s 89ms/step - loss: 260.5229 - val_loss: 359.1539 - val_nasa: 311440.2500 - learning_rate: 0.0010
Epoch 5/60
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 246.3505 — val_nasa: 316,605.0

In [11]:
# =========================
# 10. 평가 + (핵심 v3) 캘리브레이션 (FD002: high-RUL under-pred 방지 포함)
#   - val을 pseudo-test(조기 종료)로 만들어 unit-level 캘리브레이션을 학습
#   - op_cluster(op_cond) 호환
#   - MAX_RUL=200 기준으로 일관
# =========================
import numpy as np

# --- 안전: op_cluster 컬럼 보장 ---
if 'op_cluster' not in train_df.columns:
    if 'op_cond' in train_df.columns:
        train_df['op_cluster'] = train_df['op_cond']
    else:
        raise KeyError("train_df에 op_cluster/op_cond 컬럼이 없습니다. cell 4 실행 확인.")
if 'op_cluster' not in test_df.columns:
    if 'op_cond' in test_df.columns:
        test_df['op_cluster'] = test_df['op_cond']
    else:
        raise KeyError("test_df에 op_cluster/op_cond 컬럼이 없습니다. cell 4 실행 확인.")

MAX_RUL = 200  # FD002 test truth max=194 → 200으로 확장
if 'TEST_LAST_K' not in globals():
    TEST_LAST_K = 7

def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    d = y_pred - y_true
    s = np.where(d < 0, np.exp(-d/13.0) - 1.0, np.exp(d/10.0) - 1.0)
    return float(np.sum(s))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean(np.abs(y_true - y_pred)))

# --- unit별 last-k 집계 ---
def aggregate_lastk(pred_k, units_k):
    out_units = np.unique(units_k)
    med, st = [], []
    for u in out_units:
        vals = pred_k[units_k == u]
        med.append(np.median(vals))
        st.append(np.std(vals))
    return out_units.astype(int), np.array(med, np.float32), np.array(st, np.float32)

# --- val pseudo-test 생성: 마지막보다 gap만큼 앞에서 끊긴 시점의 RUL을 정답으로 사용 ---
rng = np.random.RandomState(2026)
K_VAL = int(min(TEST_LAST_K, 7))

groups_val_np = np.asarray(groups_val)
cycles_val_np = np.asarray(cycles_val)

y_val_pred_all = model.predict(X_val, verbose=0).reshape(-1)
y_val_pred_all = np.clip(y_val_pred_all, 0, MAX_RUL)

def make_pseudo_test_targets(y_true_all, y_pred_all, units, cycles, last_k=7,
                             min_cut_gap=20, max_cut_gap=80):
    y_true_all = np.asarray(y_true_all).reshape(-1)
    y_pred_all = np.asarray(y_pred_all).reshape(-1)
    units = np.asarray(units)
    cycles = np.asarray(cycles)

    out_u = np.unique(units)
    y_t, med, st = [], [], []

    for u in out_u:
        idx = np.where(units == u)[0]
        idx_sorted = idx[np.argsort(cycles[idx])]
        max_c = float(cycles[idx_sorted[-1]])

        gap = int(rng.randint(min_cut_gap, max_cut_gap + 1))
        cut = max(1.0, max_c - gap)

        idx_cut = idx_sorted[cycles[idx_sorted] <= cut]
        if len(idx_cut) == 0:
            idx_cut = idx_sorted[:1]

        last_i = idx_cut[-1]
        y_t.append(float(y_true_all[last_i]))

        idx_lastk = idx_cut[-last_k:] if len(idx_cut) >= last_k else idx_cut
        vals = y_pred_all[idx_lastk]
        med.append(float(np.median(vals)))
        st.append(float(np.std(vals)))

    return out_u.astype(int), np.array(y_t, np.float32), np.array(med, np.float32), np.array(st, np.float32)

val_units, y_val_pseudo, val_med, val_std = make_pseudo_test_targets(
    y_val, y_val_pred_all, groups_val_np, cycles_val_np, last_k=K_VAL,
    min_cut_gap=20, max_cut_gap=80
)

# --- val unit별 마지막 op_cluster (train_df에서 추출) ---
def unit_last_cluster_for_units(df, units, op_col='op_cluster'):
    units = [int(u) for u in np.asarray(units).tolist()]
    df2 = df[df['unit_number'].astype(int).isin(units)].copy()
    last_c = {}
    for u, g in df2.groupby(df2['unit_number'].astype(int)):
        g = g.sort_values('time_cycles')
        last_c[int(u)] = int(g[op_col].iloc[-1])
    return last_c

val_cluster_map = unit_last_cluster_for_units(train_df, val_units, op_col='op_cluster')
val_clusters = np.array([val_cluster_map.get(int(u), 0) for u in val_units], dtype=np.int32)
cluster_ids = np.unique(val_clusters).tolist()

# --- 캘리브레이션: (1) std 보수화는 약하게(under-pred 방지) (2) shift는 ± 허용 (3) cap은 200 기준 ---
def apply_calibration(med, std, clusters, alpha, shifts, cmax, high_rul_guard=None):
    pred = med - alpha * std
    pred = pred + np.array([shifts.get(int(c), 0.0) for c in clusters], dtype=np.float32)
    if high_rul_guard is not None:
        # high_rul_guard = (threshold, min_ratio) : pred < min_ratio*med 인 경우 완화
        th, ratio = high_rul_guard
        pred = np.where(med >= th, np.maximum(pred, ratio * med), pred)
    pred = np.clip(pred, 0, cmax)
    return pred

alpha_grid = np.linspace(0.0, 1.0, 9)     # 0.0~1.0 (과도한 보수화 방지)
cmax_grid  = [160, 180, 200]
shift_grid = np.linspace(-15, 15, 13)     # under/over 둘 다 대응
guard_grid = [(None, None),
              (140, 0.85),
              (160, 0.80)]               # high-RUL under-pred 방지

best_cfg = {'score': float('inf')}

for cmax in cmax_grid:
    for alpha in alpha_grid:
        for guard in guard_grid:
            high_guard = None if guard[0] is None else guard
            shifts = {cid: 0.0 for cid in cluster_ids}

            # 그리디 shift 탐색
            cur_score = nasa_score(y_val_pseudo, apply_calibration(val_med, val_std, val_clusters, alpha, shifts, cmax, high_guard))
            improved = True
            while improved:
                improved = False
                for cid in cluster_ids:
                    best_local = (cur_score, shifts[cid])
                    for sh in shift_grid:
                        tmp = dict(shifts)
                        tmp[cid] = float(sh)
                        pred = apply_calibration(val_med, val_std, val_clusters, alpha, tmp, cmax, high_guard)
                        sc = nasa_score(y_val_pseudo, pred)
                        if sc < best_local[0]:
                            best_local = (sc, float(sh))
                    if best_local[0] + 1e-6 < cur_score:
                        shifts[cid] = best_local[1]
                        cur_score = best_local[0]
                        improved = True

            if cur_score < best_cfg['score']:
                best_cfg = {
                    'score': float(cur_score),
                    'alpha': float(alpha),
                    'cmax': int(cmax),
                    'shifts': shifts,
                    'high_guard': high_guard
                }

print("✅ best calibration on VAL-pseudo (NASA):", best_cfg)

val_pred_cal = apply_calibration(val_med, val_std, val_clusters, best_cfg['alpha'], best_cfg['shifts'], best_cfg['cmax'], best_cfg['high_guard'])
print("\nVAL-pseudo (unit) metrics")
print("RMSE      :", f"{rmse(y_val_pseudo, val_pred_cal):.4f}")
print("MAE       :", f"{mae(y_val_pseudo, val_pred_cal):.4f}")
print("NASA Score:", f"{nasa_score(y_val_pseudo, val_pred_cal):,.2f}")

✅ best calibration on VAL-pseudo (NASA): {'score': 90.00991821289062, 'alpha': 1.0, 'cmax': 160, 'shifts': {0: -10.0, 1: -10.0, 2: -15.0, 3: -10.0, 4: -12.5, 5: -10.0}, 'high_guard': None}

VAL-pseudo (unit) metrics
RMSE      : 11.3606
MAE       : 9.5630
NASA Score: 90.01


In [13]:
# =========================
# 11. OFFLINE 평가 + 제출 파일 생성 (FINAL SAFE)
# =========================
import numpy as np
import pandas as pd

# --- submission_df 보장 ---
if 'submission_df' not in globals():
    submission_df = pd.DataFrame({
        'unit_number': np.sort(test_df['unit_number'].unique()).astype(int)
    })

# --- OFFLINE 점수 출력 (정답 있을 때만) ---
if 'y_test_true' in globals():
    print("RAW NASA:", f"{nasa_score_total(y_test_true, y_test_pred_raw):,.2f}")
    print("CAL NASA:", f"{nasa_score_total(y_test_true, y_test_pred_cal):,.2f}")
    print("BEST NASA:", f"{nasa_score_total(y_test_true, y_test_pred):,.2f}")
else:
    print("ℹ️ y_test_true가 없어서 offline 점수 출력은 생략했어(캐글 제출이면 정상).")

# --- 제출 파일 생성 ---
sub = submission_df.copy()
sub['RUL'] = y_test_pred.astype(float)
sub.to_csv('submission.csv', index=False)
print("✅ submission.csv saved")


RAW NASA: 5,558.17
CAL NASA: 10,033.63
BEST NASA: 5,558.17
✅ submission.csv saved
